In [ ]:
import sys
import json
import re
from pathlib import Path
from collections import defaultdict
import difflib
import pandas as pd
import torch
from spacy import displacy
from IPython.display import display, HTML
from gliner import GLiNER
# Agrega la raíz del proyecto al path para importar src/
PROJECT_ROOT = Path("../").resolve()  # ajusta si se corre desde otro lugar
sys.path.insert(0, str(PROJECT_ROOT))

from src.ner.gliner_ner import GlinerNER, DocumentResult

import warnings
warnings.filterwarnings("ignore")
# 3. Parámetros
GLINER_MODEL = str(PROJECT_ROOT / "models" / "gliner_entrevistas_finetuned")
THRESHOLD    = 0.40
ENCODING     = "utf-8"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

In [ ]:
# ⏳ Primera vez: descarga ~500MB del modelo desde HuggingFace
# Las siguientes ejecuciones usan el caché local (~/.cache/huggingface)
GLINER_MODEL    = "urchade/gliner_multi_pii-v1"
THRESHOLD       = 0.40  # ajusta en §3 según los resultados
ENCODING        = "utf-8"
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

gliner = GlinerNER(
    model     = GLINER_MODEL,
    threshold = THRESHOLD,  # <-- Lo subimos a 0.55 para evitar falsos positivos
    labels    = ['persona', 'lugar', 'organizacion']
)
print(gliner.model_info())

In [ ]:
ORIGINAL_DIR = PROJECT_ROOT / "data" / "raw" / "fine_tunning"
GROUND_TRUTH_DIR = PROJECT_ROOT / "data" / "ground_truth" / "entrevistas_anotadas"
OUTPUT_DATASET = PROJECT_ROOT / "data" / "processed" / "gliner_train_dataset.json"

In [ ]:
# --- PASO 1: TRADUCTOR ESTRICTO DE ETIQUETAS ---
def clasificar_etiqueta(texto_gt):
    """
    Filtra y traduce las etiquetas del texto original a las 3 maestras.
    Si encuentra eventos u otras cosas, devuelve None para ignorarlos.
    """
    texto = texto_gt.lower()
    
    if "person" in texto:
        return "PERSONA"
    elif "university" in texto or "company" in texto or "org" in texto:
        return "ORGANIZACION"
    elif "place" in texto or "city" in texto or "country" in texto:
        return "LUGAR"
    else:
        # Ignoramos Eventos y Misceláneos
        return None

In [ ]:
# --- PASO 2: PROCESAMIENTO GLOBAL Y CHUNKING ---
def procesar_documento_completo(texto_orig, texto_gt, max_words=250):
    palabras_orig = texto_orig.split()
    palabras_gt = texto_gt.split()
    
    # 1. Versiones limpias solo para que difflib haga el "match" sin estorbo de comas
    orig_limpias = [re.sub(r'[\.,\?\:¿!¡]', '', p).lower() for p in palabras_orig]
    gt_limpias = [re.sub(r'[\.,\?\:¿!¡]', '', p).lower() for p in palabras_gt]
    
    comparador = difflib.SequenceMatcher(None, orig_limpias, gt_limpias, autojunk=False)
    entidades_globales = []
    
    # 2. Buscar coincidencias en todo el texto
    for tag, i1, i2, j1, j2 in comparador.get_opcodes():
        if tag == 'replace':
            texto_gt_reemplazo = " ".join(palabras_gt[j1:j2])
            etiquetas_encontradas = re.findall(r'\*\*.*?\*\*', texto_gt_reemplazo.lower())
            
            if etiquetas_encontradas:
                label_limpio = clasificar_etiqueta(etiquetas_encontradas[0])
                
                # ¡Solo guardamos si pertenece a nuestras 3 categorías!
                if label_limpio is not None:
                    # Guardamos índices globales inclusivos (i2 - 1)
                    entidades_globales.append([i1, i2 - 1, label_limpio])

    # 3. Cortar el texto en bloques (chunks) para GLiNER
    dataset_doc = []
    for i in range(0, len(palabras_orig), max_words):
        chunk_start = i
        chunk_end = i + max_words
        chunk_words = palabras_orig[chunk_start:chunk_end]
        chunk_ents = []
        
        # Recalcular coordenadas para el bloque actual
        for ent_start, ent_end, label in entidades_globales:
            # Si la entidad cae completa dentro de este bloque de 250 palabras
            if ent_start >= chunk_start and ent_end < chunk_end:
                nuevo_inicio = ent_start - chunk_start
                nuevo_fin = ent_end - chunk_start
                chunk_ents.append([nuevo_inicio, nuevo_fin, label])
                
        # Solo guardamos el bloque de entrenamiento si tiene entidades
        if len(chunk_ents) > 0:
            dataset_doc.append({
                "tokenized_text": chunk_words, # Guardamos el original (con mayúsculas)
                "ner": chunk_ents
            })
            
    return dataset_doc

In [ ]:
# --- PASO 3: BUCLE PRINCIPAL (FILTRADO C1 Y C2) ---
import json

dataset_entrenamiento = []
dataset_evaluacion = []

archivos_originales = sorted(ORIGINAL_DIR.glob("*.txt"))

print("Construyendo datasets de Entrenamiento y Evaluación...")

for ruta_orig in archivos_originales:
    doc_id = ruta_orig.stem
    ruta_gt = GROUND_TRUTH_DIR / f"{doc_id}.txt"
    
    if not ruta_gt.exists():
        print(f"Faltan anotaciones para {doc_id}, saltando...")
        continue
        
    texto_orig = ruta_orig.read_text(encoding='utf-8')
    texto_gt = ruta_gt.read_text(encoding='utf-8')
    
    # Procesar en bloques
    bloques = procesar_documento_completo(texto_orig, texto_gt)
    
    # EL FILTRO EXACTO PARA C1 Y C2
    # Convertimos todo a minúsculas por si acaso hay un error de tipeo
# 🛡️ EL FILTRO PARA TODOS LOS DOCUMENTOS (C1 a C6)
    doc_id_limpio = doc_id.lower()
    
    # Lista de los documentos que queremos en la bóveda de entrenamiento
    docs_entrenamiento = ["c1", "c2", "c3", "c4", "c5", "c6"]
    
    # Si el doc_id contiene alguno de los nombres de la lista, va a entrenamiento
    if any(doc in doc_id_limpio for doc in docs_entrenamiento):
        dataset_entrenamiento.extend(bloques)
        print(f" ✅ {doc_id} -> Asignado a ENTRENAMIENTO (El libro de texto)")
    else:
        dataset_evaluacion.extend(bloques)
        print(f" 🧪 {doc_id} -> Asignado a PRUEBA (El examen final)")

# 1. Guardar el JSON de Entrenamiento (C1 y C2)
OUTPUT_TRAIN = PROJECT_ROOT / "data" / "processed" / "gliner_train_dataset.json"
OUTPUT_TRAIN.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_TRAIN, "w", encoding="utf-8") as f:
    json.dump(dataset_entrenamiento, f, ensure_ascii=False, indent=4)

# 2. Guardar el JSON de Evaluación (Todas las demás)
OUTPUT_EVAL = PROJECT_ROOT / "data" / "processed" / "gliner_eval_dataset.json"
with open(OUTPUT_EVAL, "w", encoding="utf-8") as f:
    json.dump(dataset_evaluacion, f, ensure_ascii=False, indent=4)

print(f"\nRESUMEN FINAL:")
print(f" - Bloques generados para ENTRENAR: {len(dataset_entrenamiento)}")
print(f" - Bloques generados para EVALUAR:  {len(dataset_evaluacion)}")
print(f" Guardados en: {OUTPUT_TRAIN.parent}")

In [ ]:
from gliner import GLiNER

In [ ]:
# 1. Cargar el modelo base
print("Cargando modelo base de GLiNER...")
model = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")

# 2. Cargar tus datos
ruta_dataset = PROJECT_ROOT / "data" / "processed" / "gliner_train_dataset.json"
with open(ruta_dataset, "r", encoding="utf-8") as f:
    full_data = json.load(f)

# Asignamos el 100% de los datos a entrenamiento
train_data = full_data

print(f"Muestras para entrenamiento: {len(train_data)}")
print("No se separaron muestras para evaluación (100% entrenamiento).")

# 3. Entrenar directamente con la API oficial de GLiNER
print("Iniciando entrenamiento... ")
trainer = model.train_model(
    train_dataset=train_data,
    eval_dataset=None,                 # <- Cambiado a None porque ya no hay datos de evaluación
    output_dir=str(PROJECT_ROOT / "models" / "checkpoints"),
    learning_rate=5e-6,
    others_lr=1e-5,
    per_device_train_batch_size=8,     # Reduce a 4 o 2 si da error OOM en GPU
    num_train_epochs=5,
    save_steps=100,
    save_total_limit=2,
    dataloader_num_workers=0
)

# 4. Guardar el modelo entrenado
ruta_modelo_final = str(PROJECT_ROOT / "models" / "gliner_entrevistas_finetuned")
model.save_pretrained(ruta_modelo_final)

print("¡Entrenamiento completado exitosamente!")
print(f"Modelo final guardado en: {ruta_modelo_final}")

In [ ]:
# Guardar el modelo en su punto óptimo (Paso 200 / ~12 épocas)
ruta_modelo_final = str(PROJECT_ROOT / "models" / "gliner_entrevistas_finetuned")
model.save_pretrained(ruta_modelo_final)

print("¡Modelo guardado en su punto óptimo!")
print(f"Guardado en: {ruta_modelo_final}")